# Figure 4: Drought resistance

Standalone paired-array workflow. The final cell retains the plotting logic from public `Main_figure.ipynb` cell 33. The optional protected-area background is a display-only 0.01-degree mask cropped to the map extent; it does not enter the scientific calculations.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "config" / "analysis.json").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise FileNotFoundError("Could not locate the analysis repository root")
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from elephant_analysis.runtime import configure_runtime
configure_runtime()

import numpy as np
import matplotlib.pyplot as plt
from elephant_analysis.data_inputs import load_inputs
from elephant_analysis.artifacts import save_arrays, load_arrays
from elephant_analysis.array_utils import nanravel
from elephant_analysis.paired_sites import compute_paired

DATA_ROOT = REPO_ROOT / "data"
RESULTS_ROOT = REPO_ROOT / "results"
FIGURES_ROOT = REPO_ROOT / "figures"
FIGURES_ROOT.mkdir(parents=True, exist_ok=True)
INPUT_MODE = "reference"
RUN_ANALYSIS = True


In [ ]:
cache_path = RESULTS_ROOT / "paired" / "paired_results.npz"
if INPUT_MODE not in ("reference", "raw"):
    raise ValueError("INPUT_MODE must be reference or raw")
if RUN_ANALYSIS:
    loaded = load_inputs("paired", DATA_ROOT, mode=INPUT_MODE)
    paired_arrays = compute_paired(loaded) if INPUT_MODE == "raw" else loaded
    save_arrays(cache_path, **paired_arrays)
    paired_arrays = load_arrays(cache_path)
# Cached mode loads the figure-specific result in the next calculation cell.


## Paired inputs


In [ ]:
from elephant_analysis.figure_statistics import prepare_resistance_plot_data
plot_path = RESULTS_ROOT / "fig4/resistance_plot_data.npz"
if RUN_ANALYSIS:
    save_arrays(plot_path, **prepare_resistance_plot_data(paired_arrays))
plot_data = load_arrays(plot_path)
delta = [plot_data["map_ndvi"], plot_data["map_sif"]]
res_delta = [plot_data["delta_ndvi"], plot_data["delta_sif"]]
relative_delta = [plot_data["relative_ndvi"], plot_data["relative_sif"]]

# This display mask preserves the source 0.01-degree resolution. Zeros must
# be transparent; applying a one-colour map to unmasked zeros paints the ocean.
overlay_path = DATA_ROOT / "reference" / "protected_area_display.npz"
protected_area_display = None
if overlay_path.is_file():
    overlay = load_arrays(overlay_path)
    if not np.array_equal(overlay["extent"], [-26, 53, -40, 17]):
        raise ValueError("Protected-area display extent does not match the figure")
    if overlay["mask"].shape != (5700, 7900):
        raise ValueError("Expected the 0.01-degree display mask cropped to the map extent")
    protected_area_display = np.ma.masked_equal(overlay["mask"], 0)
else:
    print("Optional protected-area display mask absent; background overlay omitted.")


## Figure


In [ ]:
# Original plotting cells copied from Main_figure.ipynb.
# Keep source sizes, font settings, labels, colours, positions and line widths.
EXPORT_PDF = True
EXPORT_DPI = 600
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import kstest, gaussian_kde
from elephant_analysis import array_utils as gc
mpl.rcdefaults()
mpl.rcParams["pdf.fonttype"] = 42

def export_original(figure, stem, dpi=EXPORT_DPI):
    # Match the source savefig contract, including tight bounding boxes.
    FIGURES_ROOT.mkdir(parents=True, exist_ok=True)
    figure.savefig(FIGURES_ROOT / (stem + '.png'), dpi=dpi, bbox_inches='tight')
    if EXPORT_PDF:
        figure.savefig(FIGURES_ROOT / (stem + '.pdf'), bbox_inches='tight')

from mpl_toolkits.basemap import Basemap
plt.rc('font', family='sans-serif', **{'sans-serif': 'Arial', 'weight': 'normal', 'size': 14})
if protected_area_display is None:
    raise FileNotFoundError('Original Fig. 4 background requires reference/protected_area_display.npz')
# BEGIN ORIGINAL CELL 33
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib import colors
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
font = {'family': 'sans-serif',
        'sans-serif': 'Helvetica',
        'weight': 'normal',
        'size': 16}

lon1 = -26
lon2 = 53
lat1 = -40
lat2 = 17

left, bottom, width, hegiht= 0, 0.05, 0.5,1
left1, bottom1, width1, hegiht1= 0, -0.45, 0.5,1
loc=[[left, bottom, width, hegiht],[left1, bottom1, width1, hegiht1]]
left2, bottom2, width2, hegiht2= 0.65, 0.5, 0.3,0.3
left3, bottom3, width3, hegiht3= 0.65, -0.14, 0.3,0.4
loc1=[[left2, bottom2, width2, hegiht2],[left3, bottom3, width3, hegiht3]]

cl= np.array([(0,0,255),(159,159,255)])
cl=cl/255
cl1= np.array([(0,0,255),(109,109,255)])
cl1=cl1/255
res_name=['ndvi_res','csif_res']

fig = plt.figure(figsize=(12,8))
fig.text(0.02, 0.8, 'a', fontweight='bold', fontsize=24, fontname='Arial')
fig.text(0.55, 0.8, 'b', fontweight='bold', fontsize=24, fontname='Arial')
fig.text(0.02, 0.3, 'c', fontweight='bold', fontsize=24, fontname='Arial')
fig.text(0.55, 0.3, 'd', fontweight='bold', fontsize=24, fontname='Arial')
label=['$\Delta$ Resistance']
text=['a','b']
cmap = colors.ListedColormap([(197/255, 221/255, 197/255)])
bounds = [0.5, 1.5]
norm1 = colors.BoundaryNorm(bounds, cmap.N)

for i in range(2):
    ax = fig.add_subplot(loc[i])

    m = Basemap(llcrnrlon=lon1,llcrnrlat=lat1,urcrnrlon=lon2,urcrnrlat=lat2,ax=ax)
    m.drawlsmask(land_color='whitesmoke',ocean_color='none',lakes=True)
    m.drawcountries(linewidth=0.30, color='dimgrey')
    norm = mcolors.TwoSlopeNorm(vmin=-0.05, vmax = 0.05, vcenter=0)
    im = m.imshow((delta[i])[32:],cmap ='bwr_r',origin='upper',norm=norm,zorder=2)
    im1 = m.imshow(protected_area_display,cmap =cmap,origin='upper',norm=norm1,zorder=1)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['left'].set_visible(False)

    position=fig.add_axes([0.12,-0.2,0.3,0.01])
    cb=fig.colorbar(im,ax=ax,extend='both',shrink=0.3,pad=0.01,orientation='horizontal',cax=position)
    cb.set_label(label[0],fontsize=18,labelpad=10)
    cb.ax.tick_params(labelsize=16) 
    cb.set_ticks(np.arange(-0.05,0.051,0.05)) 
    cb.outline.set_visible(False)

    ax1=fig.add_subplot(loc1[0])
    x=np.linspace(-1,1,1000)
    mean=np.nanmean(res_delta[i])
    delta_rav=gc.nanravel(res_delta[i])
    kenal=gaussian_kde(delta_rav)
    z=kenal.evaluate(x)
    z_mean=kenal.evaluate(mean)
    print(mean,z_mean)

    xlabel=['$\Delta$Resistance','$\Delta$Resistance_SIF']
    ax1.plot(x,z,lw=3,color=cl[i])
    ax1.vlines(mean,0,z_mean,lw=2,ls='--',color=cl[i])
    ax1.vlines(0,0,25,lw=2,ls='--',color='black')
    ax1.spines['right'].set_visible(False)
    ax1.spines['top'].set_visible(False)
    p=plot_data['delta_ks_p'][i]  # Use saved resistance diagnostic, avoiding the stale temperature variable.
    print("delta res, mean=",mean,"p=",p, "N=", len(delta_rav))
    print("delta res .s.e",stats.sem(delta_rav))
    if p<0.001:
        ax1.text(0.07,0.85-i*0.1, 'p<0.001', fontsize=14,transform = ax1.transAxes,color=cl[i])
    elif p<0.01:
        ax1.text(0.07,0.85-i*0.1, 'p<0.01', fontsize=14,transform = ax1.transAxes,color=cl[i])
    elif p<0.05:
        ax1.text(0.07,0.85-i*0.1, 'p<0.05', fontsize=14,transform = ax1.transAxes,color=cl[i])
    else:
        ax1.text(0.07,0.85-i*0.1, 'p={:.2f}'.format(p), fontsize=14,transform = ax1.transAxes,color=cl[i]) 
    
    ax1=plt.gca()
    ax1.yaxis.set_ticks_position('left')
    ax1.spines['left'].set_position(('data',-0.055))
    ax1.xaxis.set_ticks_position('bottom')
    ax1.spines['bottom'].set_position(('data',-4.5))
    ax1.set_ylim(0,18)
    ax1.set_xlim(-0.05,0.05)
    ax1.set_xticks(np.arange(-0.05,0.051,0.05))
    ax1.set_yticks(np.arange(0,20.1,10))
    ax1.set_ylabel('Probability density',labelpad=15)
    ax1.set_xlabel(xlabel[0],labelpad=15)
    ax1.tick_params(labelsize=16)
    ax1.xaxis.label.set_size(18)
    ax1.yaxis.label.set_size(18)
    ax1.xaxis.set_tick_params(width=2)
    ax1.yaxis.set_tick_params(width=2)
    ax1.spines['left'].set_linewidth(2)
    ax1.spines['bottom'].set_linewidth(2)
    ax1.set(facecolor='none')
    pos = ax1.get_position()
    if i ==0:
        ax2 = fig.add_axes([pos.x0, pos.y0 - 0.05, pos.width, 0.06],sharex=ax1)

        ax2.text(-0.0,0.325, "NDVI", fontsize=14,transform = ax2.transAxes,color=cl[i])
    else:
        ax2 = fig.add_axes([pos.x0, pos.y0 - 0.08, pos.width, 0.06],sharex=ax1)

        ax2.text(-0.0,0.32, "SIF", fontsize=14,transform = ax2.transAxes,color=cl[i])
    ax2.axis('off')
    ax2.set_xlim(-0.05,0.05)
    ax2.set_xticks(np.arange(-0.05,0.0501,0.05))
    ax2.spines['top'].set_visible(False)
    ax2.spines['bottom'].set_visible(True)
    bplot=ax2.boxplot(res_delta[i],
                      vert=False,
                      whis=(10,90),
                      widths=0.3,
                      patch_artist=True,
                      showmeans=False,
                      meanprops = {'marker':'o','markerfacecolor':'C0',"markeredgecolor":'C0',"markersize":15,"alpha":0},
                      medianprops={'linewidth':'0.5',"color":'C0',"alpha":0},
                      boxprops={"facecolor": 'none', "edgecolor": cl[i],"linewidth":2,"alpha":1},
                      capprops=None,
                      whiskerprops={'linewidth':'2','color':cl[i]},
                      showcaps=False,
                      showfliers = False,
                      flierprops = {'marker':'+','markersize':'3','markeredgecolor':cl[i],'color':cl[i]})
    ax2.set(facecolor='none')

    ax3=fig.add_subplot(loc1[1])
    ps=[[1],[1.5]]
    x1=[1,1.5]
    x2=[1.25]
    for i in range(2):
        ax3.boxplot(relative_delta[i],
                    vert=True,
                    whis=(10,90),                
                    widths=0.25,
                    patch_artist=True,
                    showmeans=False,
                    meanprops = {'marker':'o','markerfacecolor':'blue',"markeredgecolor":'blue',"markersize":15,"alpha":0},
                    medianprops={'linewidth':'2',"color":cl[i],"alpha":1},
                    boxprops={"facecolor": cl[i], "edgecolor": cl[i],"linewidth":2,"alpha":0.3},
                    capprops=None,
                    whiskerprops={'linewidth':'2','color':cl[i]},
                    showcaps=False,
                    showfliers = False,
                    flierprops = {'marker':'+','markersize':'3','markeredgecolor':cl[i],'color':cl[i]},
                    positions=ps[i])
        patch=mpatches.Patch(color=cl[i],label='Short')
        ax3.scatter(x1[i],np.mean(relative_delta[i]),color= cl[i],s=130)
        ax3.hlines(0,0.75,1.75,lw=2,ls='--',color='black')
        p=kstest(relative_delta[i], 'norm')[1]
        print("relative res reduction, mean=", np.mean(relative_delta[i]), "p=", p, "N=", len(delta_rav))
        print("relative res reduction .s.e",stats.sem(relative_delta[i]))
        if p<0.001:
            ax3.text(0.07,0.85-i*0.08, 'p<0.001', fontsize=14,transform = ax3.transAxes,color=cl[i])
        elif p<0.01:
            ax3.text(0.07,0.85-i*0.08, 'p<0.01', fontsize=14,transform = ax3.transAxes,color=cl[i])
        elif p<0.05:
            ax3.text(0.07,0.85-i*0.08, 'p<0.05', fontsize=14,transform = ax3.transAxes,color=cl[i])
        else:
            ax3.text(0.07,0.85-i*0.08, 'p={:.2f}'.format(p), fontsize=14,transform = ax3.transAxes,color=cl[i]) 

    labels = ['NDVI', 'SIF']
    ax3.spines['top'].set_visible(False)
    ax3.spines['right'].set_visible(False)
    ax3.spines['bottom'].set_visible(False)
    ax3.set_xticks(x1,labels=labels,rotation=0)
    ax3.xaxis.set_tick_params(labelsize=18,pad=10)
    ax3.yaxis.set_tick_params(labelsize=15)
    ax3.tick_params(bottom=True,top=False, left=True, right=False)

    ax3.set_ylabel('PAL (%)',size=18,labelpad=15)
    ax3.set_ylim(-150,200)
    ax3.set_yticks(np.arange(-150,201,50))

    ax3.xaxis.set_tick_params(width=2)
    ax3.spines['left'].set_linewidth(2)
# END ORIGINAL CELL 33
export_original(fig, 'Fig4_drought_resistance', dpi=300)
plt.show()
